In [1]:
# Install the required dependencies 
!pip install scikit-learn prettytable

In [2]:
# Clone the official BN-WVAD repository.
!git clone https://github.com/cool-xuan/BN-WVAD.git
%cd BN-WVAD
print("clone executed")

Cloning into 'BN-WVAD'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 47 (delta 6), reused 5 (delta 5), pack-reused 30 (from 1)
Receiving objects: 100% (47/47), 20.90 MiB | 37.41 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/kaggle/working/BN-WVAD
clone executed


In [3]:
!ls /kaggle/working/BN-WVAD

ckpts		   imgs      logs     models	  test.py
dataset_loader.py  infer.py  losses   options.py  train.py
frame_label	   list      main.py  README.md   utils.py


In [4]:
# ============================================================
# Dataset directory initialization
# ------------------------------------------------------------
# The BN-WVAD framework expects a specific directory structure
# for features, video lists, and frame-level annotations.
#
# The following folders are created to match the expected
# layout for the XD-Violence dataset during inference.
# ============================================================
!mkdir -p data_root/XD-Violence/features/test
!mkdir -p data_root/XD-Violence/list
!mkdir -p data_root/XD-Violence/frame_labels


In [6]:
%%writefile /kaggle/working/BN-WVAD/infer.py
#Inference script for BN-WVAD on the XD-Violence dataset.
import torch
import numpy as np
from dataset_loader import XDVideo
from options import parse_args
import pdb
import utils
import os
from models import WSAD
from tqdm import tqdm
from dataset_loader import data
from sklearn.metrics import roc_curve,auc,precision_recall_curve
import prettytable

def get_predict(test_loader, net):
    """
    Generates frame-level anomaly predictions from the test set.

    For each batch of video clips:
    - the model produces clip-level anomaly scores
    - scores are temporally expanded to frame-level by repeating
      each clip prediction over 16 frames (I3D temporal resolution)

    The resulting frame-level predictions are saved to disk
    and returned as a NumPy array.
    """
    load_iter = iter(test_loader)
    frame_predict = []
        
    for i in range(len(test_loader.dataset)//5):
        _data, _label = next(load_iter)
        
        _data = _data.cuda()
        _label = _label.cuda()
        res = net(_data)   
        
        a_predict = res.cpu().numpy().mean(0)   

        fpre_ = np.repeat(a_predict, 16)
        frame_predict.append(fpre_)

    frame_predict = np.concatenate(frame_predict, axis=0)
    np.save("/kaggle/working/BN-WVAD/frame_label/pred.npy", frame_predict)
    return frame_predict

def get_sub_metrics(frame_predict, frame_gt):
    """
    Computes evaluation metrics on the anomalous subset only.

    A predefined anomaly mask is used to select frames belonging
    to anomalous temporal regions. This evaluation follows the
    standard XD-Violence protocol and allows fair comparison
    with prior work.
    """
    anomaly_mask = np.load('frame_label/xd_anomaly_mask.npy')
    sub_predict = frame_predict[anomaly_mask]
    sub_gt = frame_gt[anomaly_mask]
    
    fpr,tpr,_ = roc_curve(sub_gt, sub_predict)
    auc_sub = auc(fpr, tpr)

    precision, recall, th = precision_recall_curve(sub_gt, sub_predict)
    ap_sub = auc(recall, precision)
    return auc_sub, ap_sub

def get_metrics(frame_predict, frame_gt):
    """
    Computes full-dataset and subset evaluation metrics.

    Metrics include:
    - ROC-AUC and AP over all frames
    - ROC-AUC_sub and AP_sub computed only on anomalous frames

    Returned values are later reported as percentages.
    """
    metrics = {}
    
    fpr,tpr,_ = roc_curve(frame_gt, frame_predict)
    metrics['AUC'] = auc(fpr, tpr)
    
    precision, recall, th = precision_recall_curve(frame_gt, frame_predict)
    metrics['AP'] = auc(recall, precision)

    auc_sub, ap_sub = get_sub_metrics(frame_predict, frame_gt)
    metrics['AUC_sub'] = auc_sub
    metrics['AP_sub'] = ap_sub

    return metrics

def test(net, test_loader, test_info, step, model_file = None):
    """
    Runs inference on the test set and computes evaluation metrics.

    The function loads a pre-trained model, generates frame-level
    predictions, computes metrics, and stores them in a structured
    dictionary for logging and reporting.
    """
    with torch.no_grad():
        net.eval()
        net.flag = "Test"
        if model_file is not None:
            net.load_state_dict(torch.load(model_file))

        frame_gt = np.load("frame_label/xd_gt.npy")
        
        frame_predict = get_predict(test_loader, net)

        metrics = get_metrics(frame_predict, frame_gt)
        
        test_info['step'].append(step)
        for score_name, score in metrics.items():
            metrics[score_name] = score * 100
            test_info[score_name].append(metrics[score_name])

        return metrics


if __name__ == "__main__":
    args = parse_args()
    if args.debug:
        pdb.set_trace()
    worker_init_fn = None
    if args.seed >= 0:
        utils.set_seed(args.seed)
        worker_init_fn = np.random.seed(args.seed)
    net = WSAD(args.len_feature, flag = "Test", args = args)
    net = net.cuda()
    test_loader = data.DataLoader(
        XDVideo(root_dir = args.root_dir, mode = 'Test', num_segments = args.num_segments, len_feature = args.len_feature),
            batch_size = 5,
            shuffle = False, num_workers = args.num_workers,
            worker_init_fn = worker_init_fn)
    
    test_info = {'step': [], 'AUC': [], 'AUC_sub': [], 'AP': [], 'AP_sub': []}

    res = test(net, test_loader, test_info, 1, model_file = args.model_path)

    pt = prettytable.PrettyTable()
    pt.field_names = ['AUC', 'AUC_sub', 'AP', 'AP_sub']
    for k, v in res.items():
        res[k] = round(v, 2)
    pt.add_row([res['AUC'], res['AUC_sub'], res['AP'], res["AP_sub"]])
    print(pt)

Overwriting /kaggle/working/BN-WVAD/infer.py


In [7]:
!cp list/XD_Test.list data_root/XD-Violence/list/

In [8]:
import os
import shutil
# ============================================================
# Dataset list adaptation (environment-dependent)
# ------------------------------------------------------------
# The XD_Test.list file contains absolute paths to pre-extracted
# feature files and is therefore environment-specific.
#
# Since Kaggle uses a different filesystem structure, the list
# is reused without modification of labels or ordering, while
# feature files are copied locally to match the expected paths.
#
# This operation does NOT alter:
# - video ordering
# - annotations
# - evaluation protocol
#
# It only ensures compatibility with the execution environment.
# ============================================================

list_file = "/kaggle/working/BN-WVAD/data_root/XD-Violence/list/XD_Test.list"  
src_root = "/kaggle/input/xdv-features/i3d-features_update/i3d-features/RGBTest"  
dst_root = "/kaggle/working/BN-WVAD/data_root/XD-Violence/features/test"  
os.makedirs(dst_root, exist_ok=True)


with open(list_file, "r") as f:
    lines = sorted([line.strip() for line in f])


for l in lines:
    
    dst_fname = l.replace("test/", "")  
    dst_path = os.path.join(dst_root, dst_fname)

    
    src_fname = dst_fname.replace("#", "")
    src_path = os.path.join(src_root, src_fname)

    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)



In [9]:
!cp frame_label/* data_root/XD-Violence/frame_labels/

In [10]:
!pip install ipdb


In [11]:
!python infer.py --model_path ckpts/xd_best.pkl --root_dir /kaggle/working/BN-WVAD/data_root/XD-Violence/features


+-------+---------+-------+--------+
|  AUC  | AUC_sub |   AP  | AP_sub |
+-------+---------+-------+--------+
| 94.71 |  83.59  | 84.93 | 85.45  |
+-------+---------+-------+--------+


In [12]:
def load_video_boundaries(list_file):
    """
    Groups feature files by video and counts the number of clips
    per video.

    This information is used to reconstruct frame-level
    predictions and ground truth sequences.
    """
    paths = sorted([l.strip() for l in open(list_file)])
    video_info = {}
    for p in paths: 
        fname = p.split("/")[-1] 
        base = "__".join(fname.split("__")[:-1]) 
        if base not in video_info:
            video_info[base] = 0
        video_info[base] += 1 
    return list(video_info.items()) 


def detect_fight_videos(list_file):
    """
    Detects whether each video contains fight events (label B1)
    based on its filename annotation.

    Returns a dictionary mapping each video to a binary flag:
    1 = fight present, 0 = no fight.
    """
    paths = sorted([l.strip() for l in open(list_file)])
    fight_dict = {}
    for p in paths:
        fname = p.split("/")[-1]
        base = "__".join(fname.split("__")[:-1]) 
        if "_label_" in base: 
            after = base.split("_label_")[1]
            labels = after.split("-")
            is_fight = any(lbl == "B1" for lbl in labels)
            fight_dict[base] = int(is_fight)
        else:
            fight_dict[base] = 0
    return fight_dict 


In [13]:
def get_fight_frames(pred, gt, list_file):
    """
    Extracts frame-level predictions and ground truth for videos
    containing fight events only.

    Frames belonging to non-fight videos are zeroed out to
    ensure evaluation focuses exclusively on fight-related data.
    """
    video_info = load_video_boundaries(list_file)
    fight_dict = detect_fight_videos(list_file)

    frame_idx = 0
    fight_pred = []
    fight_gt = []

    for video_name, num_clips in video_info:
        frames = num_clips * 16
        if fight_dict[video_name] == 1:  # solo video con rissa
            fight_pred.extend(pred[frame_idx: frame_idx + frames])
            fight_gt.extend(gt[frame_idx: frame_idx + frames])
        else:
            fight_pred.extend([0] * frames)
            fight_gt.extend([0] * frames)
        frame_idx += frames

    return np.array(fight_pred), np.array(fight_gt)

In [14]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc as sk_auc

pred = np.load("/kaggle/working/BN-WVAD/frame_label/pred.npy")  
gt = np.load("/kaggle/working/BN-WVAD/frame_label/xd_gt.npy")            
fight_pred, fight_gt = get_fight_frames(pred, gt, "/kaggle/working/BN-WVAD/data_root/XD-Violence/list/XD_Test.list")
precision, recall, thresholds = precision_recall_curve(fight_gt, fight_pred)


f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

best_idx = f1_scores.argmax()
best_f1 = f1_scores[best_idx]
best_precision = precision[best_idx]
best_recall = recall[best_idx]

print("Best F1:", best_f1)
print(" Best Precision:", best_precision)
print(" Best Recall:", best_recall)

pr_auc = sk_auc(recall, precision)
print("PR-AUC:", pr_auc)

roc_auc = roc_auc_score(fight_gt, fight_pred)
print("ROC-AUC:", roc_auc)

Best F1: 0.789911037246544
 Best Precision: 0.8333333333333334
 Best Recall: 0.7507898160193273
PR-AUC: 0.8522573426244142
ROC-AUC: 0.9877076696211662


In [15]:
def get_fight_videos_for_sub(pred, gt, list_file):
    """
    Computes subset predictions and ground truth for fight videos
    that contain at least one positive (anomalous) frame.

    This constraint is required to ensure the validity of
    subset-based AUC and AP metrics.
    """
    video_info = load_video_boundaries(list_file)
    fight_dict = detect_fight_videos(list_file)

    frame_idx = 0

    sub_pred = []
    sub_gt = []

    for video_name, num_clips in video_info:
        frames = num_clips * 16

       
        if fight_dict[video_name] == 1:

            vid_pred = pred[frame_idx: frame_idx + frames]
            vid_gt   = gt[frame_idx: frame_idx + frames]

            
            if np.sum(vid_gt) > 0:
                sub_pred.extend(vid_pred)
                sub_gt.extend(vid_gt)

        frame_idx += frames

    return np.array(sub_pred), np.array(sub_gt)


In [16]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
fight_pred_sub, fight_gt_sub = get_fight_videos_for_sub(
    pred, gt, "/kaggle/working/BN-WVAD/data_root/XD-Violence/list/XD_Test.list"
)

fpr,tpr,_ = roc_curve(fight_gt_sub, fight_pred_sub)
auc_metric = auc(fpr, tpr)
print("AUC_sub (only fights videos):", auc_metric)
# --- AP_sub ---
precision_sub, recall_sub, _ = precision_recall_curve(fight_gt_sub, fight_pred_sub)
pr_auc_sub = auc(recall_sub, precision_sub)
print("PR-AUC_sub (only fights videos):", pr_auc_sub)


AUC_sub (only fights videos): 0.8126638467159327
PR-AUC_sub (only fights videos): 0.9624211823496178
